# **Install requirements**

In [1]:
!pip install gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.55.4
    Uninstalling transformers-4.55.4:
      Successfully uninstalled transformers-4.55.4


In [2]:
from gliner import GLiNER

In [3]:
# available models: https://huggingface.co/urchade

model = GLiNER.from_pretrained("urchade/gliner_mediumv2.1")
model.eval()
print("ok")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/781M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/781M [00:00<?, ?B/s]

gliner_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


ok


In [4]:
text = """
"What is the total discount amount offered during the clearance sale last quarter?"
"""

labels = ["measure", "dimension", "filter", "timeframe"]

entities = model.predict_entities(text, labels, threshold=0.4)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


last quarter => timeframe


In [ ]:
# Load the first 50 questions from the questions_only.txt file
questions = []
with open('questions_only.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 50:  # Only take first 50 questions
            break
        questions.append(line.strip())

print(f"Loaded {len(questions)} questions")
print("First question:", questions[0])

In [ ]:
import time
import json

# Define the labels for NER
labels = ["MEASURE", "DIMENSION", "DIMENSION_VALUE", "TIMEFRAME", "TIMEGRAIN", "CALCULATION", "FILTER"]

# Store results
results = []

# Process each question
for i, question in enumerate(questions):
    print(f"Processing question {i+1}/50: {question[:80]}...")
    
    # Record start time
    start_time = time.time()
    
    # Run inference
    entities = model.predict_entities(question, labels, threshold=0.4)
    
    # Record end time
    end_time = time.time()
    inference_time = end_time - start_time
    
    # Store results
    result = {
        "question": question,
        "entities": entities,
        "inference_time": inference_time
    }
    results.append(result)
    
    # Print progress
    print(f"  Found {len(entities)} entities in {inference_time:.4f}s")
    for entity in entities:
        print(f"    {entity['text']} => {entity['label']}")

print(f"\nCompleted processing {len(questions)} questions")

In [ ]:
import os

# Create the directory if it doesn't exist
os.makedirs('NER/data/processed/', exist_ok=True)

# Now save the results
output_file = 'NER/data/processed/gliner_inference_results_50_questions.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Results saved to {output_file}")

In [ ]:
# Save results to JSON file
output_file = 'NER/data/processed/gliner_inference_results_50_questions.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Results saved to {output_file}")

# Print summary statistics
total_time = sum(r['inference_time'] for r in results)
avg_time = total_time / len(results)
total_entities = sum(len(r['entities']) for r in results)

print(f"\nSummary:")
print(f"Total questions processed: {len(questions)}")
print(f"Total entities found: {total_entities}")
print(f"Total inference time: {total_time:.4f}s")
print(f"Average inference time per question: {avg_time:.4f}s")